In [ ]:
"""
코랩용 KTO (Kahneman-Tversky Optimization) 학습 스크립트
./finetuning/finetuning_data/crm-kto-dataset/cycle_01.jsonl 파일로 1 사이클 KTO 학습을 수행합니다.
./finetuning/checkpoints_kto에 Trainer 메타 데이터를 저장하고 resume을 통해 추가 학습할 수 있도록 합니다.
adapter는 /content/drive/MyDrive/멋사/adapters_kto_1에 저장합니다.
"""

In [ ]:
import torch
torch.cuda.is_available()


In [ ]:
!pip install datasets peft trl bitsandbytes accelerate
!pip install -U transformers
!pip show transformers


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
print(os.getcwd())
print(os.listdir())


In [ ]:
!git clone https://github.com/jjjh02/AmoRe_crm_generator.git
%cd AmoRe_crm_generator
!git checkout jinhyeok
!git branch
os.chdir("/content/AmoRe_crm_generator")
print(os.getcwd())


In [ ]:
from dotenv import load_dotenv
load_dotenv()


In [ ]:
import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from datasets import load_dataset, Dataset
from peft import LoraConfig, PeftModel
from trl import KTOTrainer, KTOConfig

# 모델 및 경로 설정
MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"
CACHE_DIR = "./models"
OUTPUT_DIR = "./finetuning/checkpoints_kto"
OUTPUT_ADAPTER_DIR = "/content/drive/MyDrive/LikeLion/adapters_kto_v1"
BASE_ADAPTER_PATH = "/content/drive/MyDrive/LikeLion/adapters_sft_v2"
NEW_ADAPTER_NAME = "kto_adapter_v1"

# 데이터셋 경로 설정
DATA_DIR = "/content/AmoRe_crm_generator/finetuning/finetuning_data/crm-kto-dataset"
JSON_FILE = os.path.join(DATA_DIR, "cycle_01.jsonl")

# 하이퍼파라미터 설정
MAX_SEQ_LENGTH = 1512


def load_kto_dataset(json_path: str):
    """JSON 파일에서 KTO 형식의 데이터셋을 로드합니다.

    JSON 형식:
    [
      { "prompt": "...", "chosen": "...", "rejected": "..." },
      ...
    ]

    Args:
        json_path: JSON 파일 경로

    Returns:
        train_dataset, eval_dataset
    """
    dataset = load_dataset("json", data_files=json_path)["train"]
    cols = set(dataset.column_names)

    if {"prompt", "completion", "label"}.issubset(cols):
        kto_dataset = dataset
    elif {"prompt", "response", "label"}.issubset(cols):
        kto_dataset = dataset.rename_column("response", "completion")
    elif {"prompt", "chosen", "rejected"}.issubset(cols):
        rows = []
        for ex in dataset:
            rows.append({"prompt": ex["prompt"], "completion": ex["chosen"], "label": 1})
            rows.append({"prompt": ex["prompt"], "completion": ex["rejected"], "label": 0})
        kto_dataset = Dataset.from_list(rows)
    else:
        raise ValueError(
            f"Unsupported columns: {dataset.column_names}\n"
            "Required columns are (prompt, completion, label) or (prompt, chosen, rejected)."
        )

    # train / eval split
    kto_dataset = kto_dataset.train_test_split(test_size=0.1, seed=42)

    return kto_dataset["train"], kto_dataset["test"]


def _freeze_all_params(model):
    for _, param in model.named_parameters():
        param.requires_grad = False


def _enable_adapter_params(model, adapter_name):
    for name, param in model.named_parameters():
        if f".{adapter_name}." in name:
            param.requires_grad = True


In [ ]:
"KTO 학습 메인 함수"

# 1. 토크나이저 로드
print("토크나이저 로드 중...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    cache_dir=CACHE_DIR,
)

# pad_token 설정
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 패딩 사이드 설정 (KTO 학습에 유리)
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"

# max_length 설정
tokenizer.model_max_length = MAX_SEQ_LENGTH

# 2. 데이터셋 로드
print(f"데이터셋 로드 중: {JSON_FILE}")
if not os.path.exists(JSON_FILE):
    raise FileNotFoundError(f"데이터셋 파일을 찾을 수 없습니다: {JSON_FILE}")

train_dataset, eval_dataset = load_kto_dataset(JSON_FILE)
print(f"학습 데이터: {len(train_dataset)}개, 평가 데이터: {len(eval_dataset)}개")

# 3. Flash Attention 설정
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    attn_implementation = "flash_attention_2"
    torch_dtype = torch.bfloat16
else:
    attn_implementation = "eager"
    torch_dtype = torch.float16

# 4. 모델 로드
print("모델 로드 중...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    use_cache=False,
    # attn_implementation=attn_implementation,
    torch_dtype=torch_dtype,
    cache_dir=CACHE_DIR,
)

# 5. PEFT (LoRA) 설정
print("PEFT 설정 중...")
peft_config = LoraConfig(
    lora_alpha=64,
    lora_dropout=0.05,
    r=64,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    task_type="CAUSAL_LM"
)

# 6. 베이스 어댑터 로드 (학습하지 않음)
print(f"베이스 어댑터 로드 중: {BASE_ADAPTER_PATH}")
if not os.path.exists(BASE_ADAPTER_PATH):
    raise FileNotFoundError(f"베이스 어댑터를 찾을 수 없습니다: {BASE_ADAPTER_PATH}")

model = PeftModel.from_pretrained(
    model,
    BASE_ADAPTER_PATH,
    is_trainable=True,
)

# # 7. 추가 어댑터 생성 및 활성화
# print(f"추가 어댑터 생성: {NEW_ADAPTER_NAME}")
# model.add_adapter(peft_config, NEW_ADAPTER_NAME)
# model.set_adapter(NEW_ADAPTER_NAME)
# _freeze_all_params(model)
# _enable_adapter_params(model, NEW_ADAPTER_NAME)

# 8. KTO Config 설정
print("KTO Config 설정 중...")
kto_config = KTOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=3,
    learning_rate=5e-6,
    max_grad_norm=0.3,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=1,
    logging_first_step=True,
    logging_strategy="steps",
    log_level="info",
    disable_tqdm=False,
    save_steps=80,
    save_total_limit=20,
    eval_strategy="steps",
    eval_steps=10,
    report_to="none"
)

# 9. KTOTrainer 초기화
print("KTOTrainer 초기화 중...")
trainer = KTOTrainer(
    model=model,
    ref_model=None,  # PEFT 사용 시 None으로 설정
    args=kto_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

# 10. 학습 시작
print("학습 시작...")
ckpt_dir = "AmoRe_crm_generator/finetuning/checkpoints_kto"

resume = None
if os.path.isdir(ckpt_dir) and len(os.listdir(ckpt_dir)) > 0:
    resume = True

trainer.train(resume_from_checkpoint=resume)

# 11. 모델 저장
print("모델 저장 중...")
trainer.save_model(OUTPUT_ADAPTER_DIR)
print(f"모델이 저장되었습니다: {OUTPUT_ADAPTER_DIR}")


In [ ]:
!pip install huggingface-hub


In [ ]:
# Push to HuggingFace Hub

import os

from dotenv import load_dotenv
from huggingface_hub import login, create_repo, upload_folder

login(os.getenv("HUGGINGFACE_API_KEY"))

create_repo(
    repo_id="crm-kto-adapter",
    repo_type="model",
    private=False,
    exist_ok=True
)

upload_folder(
    folder_path=OUTPUT_ADAPTER_DIR,
    repo_id="jinn33/crm-kto-adapter",
    repo_type="model",
)
